In [1]:
import pandas as pd
import numpy as np
from sklearn.ensemble import RandomForestClassifier
from sklearn.impute import SimpleImputer
from sklearn.metrics import brier_score_loss, accuracy_score
from sklearn.model_selection import train_test_split, RandomizedSearchCV
from sklearn.metrics import make_scorer
from sklearn.preprocessing import StandardScaler
import numpy as np
import numpy as np
import joblib
import os
import warnings
warnings.filterwarnings('ignore')

In [2]:
def process_avg_teamstats_data(tourney_filename, season_filename):
    # Load datasets
    tourney_results = pd.read_csv(os.path.join(data_path, tourney_filename))
    season_results = pd.read_csv(os.path.join(data_path, season_filename))

    # Calculate advanced stats separately for winners and losers
    def calculate_advanced_stats(df, team_prefix):
        stats = pd.DataFrame({
            'Season': df['Season'],
            'TeamID': df[f'{team_prefix}TeamID'],
            'Score': df[f'{team_prefix}Score'],
            'FGM': df[f'{team_prefix}FGM'],
            'FGA': df[f'{team_prefix}FGA'],
            'FGM3': df[f'{team_prefix}FGM3'],
            'FGA3': df[f'{team_prefix}FGA3'],
            'FTM': df[f'{team_prefix}FTM'],
            'FTA': df[f'{team_prefix}FTA'],
            'OR': df[f'{team_prefix}OR'],
            'DR': df[f'{team_prefix}DR'],
            'Ast': df[f'{team_prefix}Ast'],
            'TO': df[f'{team_prefix}TO']
        })
        stats["FG%"] = stats["FGM"] / stats["FGA"]
        stats["3P%"] = stats["FGM3"] / stats["FGA3"]
        stats["FT%"] = stats["FTM"] / stats["FTA"]
        stats["OR%"] = stats["OR"] / (stats["OR"] + stats["DR"])
        stats["DR%"] = stats["DR"] / (stats["OR"] + stats["DR"])
        stats["AST/TO"] = stats["Ast"] / stats["TO"]
        stats["NetRating"] = 100 * stats["Score"] / (stats["FGA"] + 0.44 * stats["FTA"] + stats["TO"])
        return stats

    # Compute for winners and losers separately
    winners_stats = calculate_advanced_stats(season_results, 'W')
    losers_stats = calculate_advanced_stats(season_results, 'L')

    # Combine all game data for each team
    all_team_stats = pd.concat([winners_stats, losers_stats])

    # Aggregate all season performance
    team_stats = all_team_stats.groupby(["Season", "TeamID"]).agg(
        GamesPlayed=("TeamID", "count"),
        Wins=("Score", lambda x: (x >= x.mean()).sum()),  # approximate wins
        AvgScore=("Score", "mean"),
        AvgFGP=("FG%", "mean"),
        Avg3PP=("3P%", "mean"),
        AvgFT=("FT%", "mean"),
        AvgORP=("OR%", "mean"),
        AvgDRP=("DR%", "mean"),
        AvgASTTO=("AST/TO", "mean"),
        AvgNetRtg=("NetRating", "mean"),
    ).reset_index()

    return team_stats


In [3]:
# Define the data path
data_path = "."

# Processing Clearly for Balanced Stats:
men_teamstats = process_avg_teamstats_data("data\MNCAATourneyDetailedResults.csv", "data\MRegularSeasonDetailedResults.csv")
women_teamstats = process_avg_teamstats_data("data\WNCAATourneyDetailedResults.csv", "data\WRegularSeasonDetailedResults.csv")

# Save the processed data to CSV files
men_teamstats.to_csv("data\Processed_Men_TeamStats.csv", index=False)
women_teamstats.to_csv("data\Processed_Women_TeamStats.csv", index=False)


In [4]:
import pandas as pd
from sklearn.ensemble import RandomForestClassifier
from sklearn.metrics import brier_score_loss, accuracy_score
from sklearn.model_selection import train_test_split, RandomizedSearchCV
from sklearn.metrics import make_scorer
import numpy as np
import warnings
warnings.filterwarnings('ignore')

def load_and_merge_data(team_stats_path, regular_season_path, tourney_path):
    team_stats = pd.read_csv(team_stats_path)
    regular_season_results = pd.read_csv(regular_season_path)
    tourney_results = pd.read_csv(tourney_path)

    # Combine regular season and tournament results
    results = pd.concat([regular_season_results, tourney_results])

    # Merge team stats for winners and losers separately
    results = results.merge(team_stats, left_on=['Season', 'WTeamID'], right_on=['Season', 'TeamID'])
    results = results.merge(team_stats, left_on=['Season', 'LTeamID'], right_on=['Season', 'TeamID'], suffixes=('', '_L'))

    # Drop the redundant 'TeamID' columns if they exist
    results.drop(columns=[col for col in ['TeamID_x', 'TeamID_y'] if col in results.columns], inplace=True)

    return results

# Merge datasets and save clearly:
men_team_stats_path = 'data\Processed_Men_TeamStats.csv'
men_regular_season_path = 'data\MRegularSeasonCompactResults.csv'
men_tourney_path = 'data\MNCAATourneyCompactResults.csv'

women_team_stats_path = 'data\Processed_Women_TeamStats.csv'
women_regular_season_path = 'data\WRegularSeasonCompactResults.csv'
women_tourney_path = 'data\WNCAATourneyCompactResults.csv'

men_merged = load_and_merge_data(men_team_stats_path, men_regular_season_path, men_tourney_path)
women_merged = load_and_merge_data(women_team_stats_path, women_regular_season_path, women_tourney_path)

final_merged = pd.concat([men_merged, women_merged])
final_merged.to_csv('final_merged.csv', index=False)

In [5]:
import pandas as pd
import numpy as np
from sklearn.impute import SimpleImputer
from sklearn.ensemble import RandomForestClassifier

# Load the data
df = pd.read_csv('final_merged.csv')

# Define columns to drop (metadata)
drop_cols = ['Season', 'DayNum', 'WTeamID', 'LTeamID', 'WScore', 'LScore', 'WLoc', 'NumOT']
features = ['GamesPlayed', 'Wins', 'AvgScore', 'AvgFGP', 'Avg3PP', 'AvgFT',
            'AvgORP', 'AvgDRP', 'AvgASTTO', 'AvgNetRtg',
            'GamesPlayed_L', 'Wins_L', 'AvgScore_L', 'AvgFGP_L', 'Avg3PP_L', 'AvgFT_L',
            'AvgORP_L', 'AvgDRP_L', 'AvgASTTO_L', 'AvgNetRtg_L']

# Initial dataset (Team1 wins)
X = df[features]
y = np.ones(len(df), dtype=int)

# Invert data (Team2 wins)
df_inv = df.copy()
for col in features:
    if col.endswith('_L'):
        base_col = col[:-2]
        df_inv[col], df_inv[base_col] = df[base_col], df[col]

X_inv = df_inv[features]
y_inv = np.zeros(len(df_inv), dtype=int)

# Combine both datasets
X_train_full = pd.concat([X, X_inv], ignore_index=True)
y_train_full = np.concatenate([y, y_inv])

# ✅ FIX: Explicitly Replace infinite values
X_train_full.replace([np.inf, -np.inf], np.nan, inplace=True)

# Check for NaNs or infinite values explicitly again
print("Infinite values present after replacement:", np.isinf(X_train_full).sum().sum())
print("NaNs before imputation:", X_train_full.isna().sum().sum())

# Impute missing values clearly
imputer = SimpleImputer(strategy='median')
X_train_full = pd.DataFrame(imputer.fit_transform(X_train_full), columns=features)

# Verify no missing or infinite values remain
assert not np.isinf(X_train_full.values).any(), "Infinite values remain!"
assert not np.isnan(X_train_full.values).any(), "NaN values remain after imputation."

# Train Random Forest clearly
rf_model = RandomForestClassifier(n_estimators=200, max_depth=7, min_samples_leaf=3, random_state=42, n_jobs=-1)
rf_model.fit(X_train_full, y_train_full)

print("✅ Model trained successfully without infinities or NaNs.")


Infinite values present after replacement: 0
NaNs before imputation: 174
✅ Model trained successfully without infinities or NaNs.


In [6]:
# Load submission_df
submission_df = pd.read_csv('data/SampleSubmissionStage2.csv')
submission_df[['Season', 'LowerID', 'HigherID']] = submission_df['ID'].str.split('_', expand=True).astype(int)

# Define team stat columns explicitly
team_stats_cols = ['GamesPlayed', 'Wins', 'AvgScore', 'AvgFGP', 'Avg3PP', 'AvgFT',
                   'AvgORP', 'AvgDRP', 'AvgASTTO', 'AvgNetRtg']

# Create a clear lookup DataFrame for team stats
winner_stats_df = df.set_index(['Season', 'WTeamID'])[team_stats_cols]
loser_stats_df = df.set_index(['Season', 'LTeamID'])[[col + '_L' for col in team_stats_cols]]
loser_stats_cols = {col + '_L': col for col in team_stats_cols}
loser_stats_df.rename(columns=loser_stats_cols, inplace=True)

# Combine winner and loser stats into one lookup DataFrame
all_team_stats_df = pd.concat([winner_stats_df, loser_stats_df])
all_team_stats_df = all_team_stats_df[~all_team_stats_df.index.duplicated(keep='first')]  # Remove duplicates clearly

# Function to safely get stats
def get_team_stats(season, team_id):
    if (season, team_id) in all_team_stats_df.index:
        stats = all_team_stats_df.loc[(season, team_id)]
        # ensure it returns a list from Series
        if isinstance(stats, pd.Series):
            return stats.tolist()
        else:  # If it's a DataFrame due to multiple matches, take the first row
            return stats.iloc[0].tolist()
    else:
        return X_train_full[team_stats_cols].median().tolist()

# Construct test set efficiently
X_test_rows = [
    get_team_stats(season, lower) + get_team_stats(season, higher)
    for season, lower, higher in submission_df[['Season', 'LowerID', 'HigherID']].itertuples(index=False)
]

X_test = pd.DataFrame(X_test_rows, columns=[f'{col}_1' for col in team_stats_cols] + [f'{col}_2' for col in team_stats_cols])

# Check the feature alignment clearly
print("✅ Features prepared for testing.")
print("Features used by model:", rf_model.feature_names_in_)
print("Test DataFrame columns:", X_test.columns.tolist())

missing_in_test = set(rf_model.feature_names_in_) - set(X_test.columns)
extra_in_test = set(X_test.columns) - set(rf_model.feature_names_in_)

print("Missing in test (should be empty):", missing_in_test)
print("Extra in test (should be empty):", extra_in_test)

# Confirm dimensions
print(X_test.shape) # Should clearly show (131407, 20)


✅ Features prepared for testing.
Features used by model: ['GamesPlayed' 'Wins' 'AvgScore' 'AvgFGP' 'Avg3PP' 'AvgFT' 'AvgORP'
 'AvgDRP' 'AvgASTTO' 'AvgNetRtg' 'GamesPlayed_L' 'Wins_L' 'AvgScore_L'
 'AvgFGP_L' 'Avg3PP_L' 'AvgFT_L' 'AvgORP_L' 'AvgDRP_L' 'AvgASTTO_L'
 'AvgNetRtg_L']
Test DataFrame columns: ['GamesPlayed_1', 'Wins_1', 'AvgScore_1', 'AvgFGP_1', 'Avg3PP_1', 'AvgFT_1', 'AvgORP_1', 'AvgDRP_1', 'AvgASTTO_1', 'AvgNetRtg_1', 'GamesPlayed_2', 'Wins_2', 'AvgScore_2', 'AvgFGP_2', 'Avg3PP_2', 'AvgFT_2', 'AvgORP_2', 'AvgDRP_2', 'AvgASTTO_2', 'AvgNetRtg_2']
Missing in test (should be empty): {'Wins', 'AvgDRP', 'AvgFT', 'AvgORP_L', 'Avg3PP_L', 'AvgNetRtg', 'Wins_L', 'AvgFGP', 'AvgFGP_L', 'Avg3PP', 'AvgScore', 'AvgFT_L', 'AvgASTTO_L', 'AvgORP', 'GamesPlayed', 'AvgASTTO', 'GamesPlayed_L', 'AvgDRP_L', 'AvgScore_L', 'AvgNetRtg_L'}
Extra in test (should be empty): {'AvgFGP_1', 'GamesPlayed_1', 'AvgNetRtg_2', 'AvgFGP_2', 'AvgDRP_2', 'AvgORP_1', 'AvgFT_2', 'AvgASTTO_2', 'Wins_1', 'Avg3PP_1', 'A

In [9]:
# Check initial infinite values and NaNs explicitly
print("Infinite values before replacement:", np.isinf(X_test.values).sum())
print("NaNs before replacement:", X_test.isna().sum().sum())

# Replace infinite values explicitly with NaN
X_test.replace([np.inf, -np.inf], np.nan, inplace=True)

# Verify intermediate NaNs after replacement
print("NaNs after replacing infinities:", X_test.isna().sum().sum())

# Explicitly rename columns BEFORE imputation to align with training data
rename_mapping = {
    'GamesPlayed_1': 'GamesPlayed',
    'Wins_1': 'Wins',
    'AvgScore_1': 'AvgScore',
    'AvgFGP_1': 'AvgFGP',
    'Avg3PP_1': 'Avg3PP',
    'AvgFT_1': 'AvgFT',
    'AvgORP_1': 'AvgORP',
    'AvgDRP_1': 'AvgDRP',
    'AvgASTTO_1': 'AvgASTTO',
    'AvgNetRtg_1': 'AvgNetRtg',
    'GamesPlayed_2': 'GamesPlayed_L',
    'Wins_2': 'Wins_L',
    'AvgScore_2': 'AvgScore_L',
    'AvgFGP_2': 'AvgFGP_L',
    'Avg3PP_2': 'Avg3PP_L',
    'AvgFT_2': 'AvgFT_L',
    'AvgORP_2': 'AvgORP_L',
    'AvgDRP_2': 'AvgDRP_L',
    'AvgASTTO_2': 'AvgASTTO_L',
    'AvgNetRtg_2': 'AvgNetRtg_L'
}
X_test.rename(columns=rename_mapping, inplace=True)

# Impute missing values explicitly using aligned columns with training median
for col in X_test.columns:
    median_val = X_train_full[col].median()
    X_test[col].fillna(median_val, inplace=True)

# Recheck explicitly to confirm no NaNs or infinities remain
assert np.isinf(X_test.values).sum() == 0, "Still contains infinities!"
assert X_test.isna().sum().sum() == 0, "Still contains NaNs!"

# Ensure data type consistency explicitly
X_test = X_test.astype(np.float64)

# Confirm feature alignment explicitly
assert set(X_test.columns) == set(rf_model.feature_names_in_), \
    f"Mismatch found: {set(rf_model.feature_names_in_) - set(X_test.columns)}"

# Generate predictions explicitly matching the model feature order
submission_df['Pred'] = rf_model.predict_proba(X_test[rf_model.feature_names_in_])[:, 1]

# Save predictions to CSV explicitly
submission_df[['ID', 'Pred']].to_csv('final_submission.csv', index=False)
print(submission_df[['ID', 'Pred']].shape)
print("✅ Final submission generated successfully: final_submission.csv")


Infinite values before replacement: 0
NaNs before replacement: 0
NaNs after replacing infinities: 0
(131407, 2)
✅ Final submission generated successfully: final_submission.csv


In [8]:
submission_df[['ID', 'Pred']].shape

(131407, 2)

In [ ]:
# Check for infinite or very large values
print("Infinite values present before replacement:", np.isinf(X_test.values).sum())
print("NaNs before imputation:", X_test.isna().sum().sum())

# Replace infinite values with NaN, then impute
X_test.replace([np.inf, -np.inf], np.nan, inplace=True)
X_test.fillna(X_train_full.median(), inplace=True)

print("Infinite values after replacement:", np.isinf(X_test.values).sum())
print("NaNs after imputation:", X_test.isna().sum().sum())

# Confirm type explicitly to match trained model
X_test = X_test.astype(np.float64)

# Recheck before prediction:
assert np.isinf(X_test.values).sum() == 0, "Still contains infinities!"
assert X_test.isna().sum().sum() == 0, "Still contains NaNs!"

# Now safely predict:
submission_df['Pred'] = rf_model.predict_proba(X_test)[:, 1]

# Save to CSV:
submission_df[['ID', 'Pred']].to_csv('final_submission.csv', index=False)
print("✅ Final submission generated successfully: final_submission.csv")


In [ ]:
# Load test dataset clearly from final merged
submission_df = pd.read_csv('data/SampleSubmissionStage2.csv')
submission_df[['Season', 'LowerID', 'HigherID']] = submission_df['ID'].str.split('_', expand=True).astype(int)

# Define team_stats_cols (exactly from training)
team_stats_cols = ['GamesPlayed', 'Wins', 'AvgScore', 'AvgFGP', 'Avg3PP', 'AvgFT',
                   'AvgORP', 'AvgDRP', 'AvgASTTO', 'AvgNetRtg']

# Create a lookup DataFrame for faster retrieval
team_stats_df = df.set_index(['Season', 'WTeamID'])[team_stats_cols]
team_stats_L_df = df.set_index(['Season', 'LTeamID'])[[col+'_L' for col in team_stats_cols]]
team_stats_L_df.columns = team_stats_cols

# Combine winner and loser stats into one lookup DataFrame
all_team_stats_df = pd.concat([team_stats_df, team_stats_L_df])

def get_team_stats(season, team_id):
    try:
        return all_team_stats_df.loc[(season, team_id)].tolist()
    except KeyError:
        return X_train_full[team_stats_cols].median().tolist()

# Vectorized predictions
X_test_rows = [
    all_team_stats_df.loc[(season, lower)].tolist() + 
    all_team_stats_df.loc[(season, higher)].tolist()
    if (season, lower) in all_team_stats_df.index and (season, higher) in all_team_stats_df.index
    else X_train_full.median().tolist() * 2
    for season, lower, higher in submission_df[['Season', 'LowerID', 'HigherID']].itertuples(index=False)
]

X_test = pd.DataFrame(X_test_rows, columns=[f'{col}_1' for col in team_stats_cols] + [col+'_2' for col in team_stats_cols])

print("✅ Test features created efficiently.")
# Check alignment explicitly
print("✅ Features used by model:", rf_model.feature_names_in_)
print("Columns in your test dataset:", X_test.columns.tolist())
print("\nColumns in training not in test:", set(rf_model.feature_names_in_) - set(X_test.columns))
print("Columns in test not in training:", set(X_test.columns) - set(rf_model.feature_names_in_))


In [ ]:
print("Features used by model:", rf_model.feature_names_in_)
print("Columns in your test dataset:", X_test.columns.tolist())


In [ ]:
# Check and fix infinite values and NaNs clearly and explicitly
print("Infinite values present before replacement:", np.isinf(X_test.values).sum())
print("NaNs before replacement/imputation:", X_test.isna().sum().sum())

# Replace infinite values with NaN
X_test.replace([np.inf, -np.inf], np.nan, inplace=True)

# Impute missing values using training set medians
X_test.fillna(X_train_full.median(), inplace=True)

# Recheck explicitly for safety
assert np.isinf(X_test.values).sum() == 0, "Still contains infinities!"
assert X_test.isna().sum().sum() == 0, "Still contains NaNs!"

# Ensure type consistency with model
X_test = X_test.astype(np.float64)

# Explicitly rename columns to align with trained model
rename_mapping = {
    'GamesPlayed_1': 'GamesPlayed',
    'Wins_1': 'Wins',
    'AvgScore_1': 'AvgScore',
    'AvgFGP_1': 'AvgFGP',
    'Avg3PP_1': 'Avg3PP',
    'AvgFT_1': 'AvgFT',
    'AvgORP_1': 'AvgORP',
    'AvgDRP_1': 'AvgDRP',
    'AvgASTTO_1': 'AvgASTTO',
    'AvgNetRtg_1': 'AvgNetRtg',
    'GamesPlayed_2': 'GamesPlayed_L',
    'Wins_2': 'Wins_L',
    'AvgScore_2': 'AvgScore_L',
    'AvgFGP_2': 'AvgFGP_L',
    'Avg3PP_2': 'Avg3PP_L',
    'AvgFT_2': 'AvgFT_L',
    'AvgORP_2': 'AvgORP_L',
    'AvgDRP_2': 'AvgDRP_L',
    'AvgASTTO_2': 'AvgASTTO_L',
    'AvgNetRtg_2': 'AvgNetRtg_L'
}
X_test.rename(columns=rename_cols, inplace=True)

# Confirm perfect feature alignment
assert set(X_test.columns) == set(rf_model.feature_names_in_), \
    f"Feature mismatch: {set(rf_model.feature_names_in_) - set(X_test.columns)}"

# Final predictions
submission_df['Pred'] = rf_model.predict_proba(X_test[rf_model.feature_names_in_])[:, 1]

# Save the submission
submission_df[['ID', 'Pred']].to_csv('final_submission.csv', index=False)
print("✅ Final submission generated: final_submission.csv")

In [ ]:
# Load test dataset clearly from final merged
submission_df = pd.read_csv('data/SampleSubmissionStage2.csv')
submission_df[['Season', 'LowerID', 'HigherID']] = submission_df['ID'].str.split('_', expand=True).astype(int)

# Define team_stats_cols (exactly from training)
team_stats_cols = ['GamesPlayed', 'Wins', 'AvgScore', 'AvgFGP', 'Avg3PP', 'AvgFT',
                   'AvgORP', 'AvgDRP', 'AvgASTTO', 'AvgNetRtg']

# Create a lookup DataFrame for faster retrieval
team_stats_df = df.set_index(['Season', 'WTeamID'])[team_stats_cols]
team_stats_L_df = df.set_index(['Season', 'LTeamID'])[[col+'_L' for col in team_stats_cols]]
team_stats_L_df.columns = team_stats_cols

# Combine winner and loser stats into one lookup DataFrame
all_team_stats_df = pd.concat([team_stats_df, team_stats_L_df])

def get_team_stats(season, team_id):
    try:
        return all_team_stats_df.loc[(season, team_id)].tolist()
    except KeyError:
        return X_train_full[team_stats_cols].median().tolist()

# Vectorized predictions
X_test_rows = [
    all_team_stats_df.loc[(season, lower)].tolist() + 
    all_team_stats_df.loc[(season, higher)].tolist()
    if (season, lower) in all_team_stats_df.index and (season, higher) in all_team_stats_df.index
    else X_train_full.median().tolist() * 2
    for season, lower, higher in submission_df[['Season', 'LowerID', 'HigherID']].itertuples(index=False)
]

X_test = pd.DataFrame(X_test_rows, columns=[f'{col}_1' for col in team_stats_cols] + [col+'_2' for col in team_stats_cols])

print("✅ Test features created efficiently.")

In [ ]:
import pandas as pd
import numpy as np
from sklearn.impute import SimpleImputer
from sklearn.ensemble import RandomForestClassifier

# Load the data
df = pd.read_csv('final_merged.csv')

# Define columns to drop (metadata)
drop_cols = ['Season', 'DayNum', 'WTeamID', 'LTeamID', 'WScore', 'LScore', 'WLoc', 'NumOT']
features = ['GamesPlayed', 'Wins', 'AvgScore', 'AvgFGP', 'Avg3PP', 'AvgFT',
            'AvgORP', 'AvgDRP', 'AvgASTTO', 'AvgNetRtg',
            'GamesPlayed_L', 'Wins_L', 'AvgScore_L', 'AvgFGP_L', 'Avg3PP_L', 'AvgFT_L',
            'AvgORP_L', 'AvgDRP_L', 'AvgASTTO_L', 'AvgNetRtg_L']

# Initial dataset (Team1 wins)
X = df[features]
y = np.ones(len(df), dtype=int)

# Invert data (Team2 wins)
df_inv = df.copy()
for col in features:
    if col.endswith('_L'):
        base_col = col[:-2]
        df_inv[col], df_inv[base_col] = df[base_col], df[col]

X_inv = df_inv[features]
y_inv = np.zeros(len(df_inv), dtype=int)

# Combine both datasets
X_train_full = pd.concat([X, X_inv], ignore_index=True)
y_train_full = np.concatenate([y, y_inv])

# ✅ FIX: Explicitly Replace infinite values
X_train_full.replace([np.inf, -np.inf], np.nan, inplace=True)

# Check for NaNs or infinite values explicitly again
print("Infinite values present after replacement:", np.isinf(X_train_full).sum().sum())
print("NaNs before imputation:", X_train_full.isna().sum().sum())

# Impute missing values clearly
imputer = SimpleImputer(strategy='median')
X_train_full = pd.DataFrame(imputer.fit_transform(X_train_full), columns=features)

# Verify no missing or infinite values remain
assert not np.isinf(X_train_full.values).any(), "Infinite values remain!"
assert not np.isnan(X_train_full.values).any(), "NaN values remain after imputation."

# Train Random Forest clearly
rf_model = RandomForestClassifier(n_estimators=200, max_depth=7, min_samples_leaf=3, random_state=42, n_jobs=-1)
rf_model.fit(X_train_full, y_train_full)

print("✅ Model trained successfully without infinities or NaNs.")


In [ ]:
from sklearn.metrics import accuracy_score, brier_score_loss
from sklearn.model_selection import train_test_split

# Perform train-validation split
X_train, X_val, y_train_split, y_val = train_test_split(
    X_train_full, y_train, test_size=0.2, random_state=42, stratify=y_train
)

# Retrain with split data
rf_model.fit(X_train, y_train_split)

# Evaluate performance
y_val_pred = rf_model.predict_proba(X_val)[:, 1]

print("Validation Set Performance:")
print(" - Accuracy:", accuracy_score(y_val, (y_val_pred >= 0.5).astype(int)))
print(" - Brier Score:", brier_score_loss(y_val, y_val_pred))


In [ ]:
# Load submission_df
submission_df = pd.read_csv('data/SampleSubmissionStage2.csv')
submission_df[['Season', 'LowerID', 'HigherID']] = submission_df['ID'].str.split('_', expand=True).astype(int)

# Define team stat columns explicitly
team_stats_cols = ['GamesPlayed', 'Wins', 'AvgScore', 'AvgFGP', 'Avg3PP', 'AvgFT',
                   'AvgORP', 'AvgDRP', 'AvgASTTO', 'AvgNetRtg']

# Create a clear lookup DataFrame for team stats
winner_stats_df = df.set_index(['Season', 'WTeamID'])[team_stats_cols]
loser_stats_df = df.set_index(['Season', 'LTeamID'])[[col + '_L' for col in team_stats_cols]]
loser_stats_cols = {col + '_L': col for col in team_stats_cols}
loser_stats_df.rename(columns=loser_stats_cols, inplace=True)

# Combine winner and loser stats into one lookup DataFrame
all_team_stats_df = pd.concat([winner_stats_df, loser_stats_df])
all_team_stats_df = all_team_stats_df[~all_team_stats_df.index.duplicated(keep='first')]  # Remove duplicates clearly

# Function to safely get stats
def get_team_stats(season, team_id):
    if (season, team_id) in all_team_stats_df.index:
        stats = all_team_stats_df.loc[(season, team_id)]
        # ensure it returns a list from Series
        if isinstance(stats, pd.Series):
            return stats.tolist()
        else:  # If it's a DataFrame due to multiple matches, take the first row
            return stats.iloc[0].tolist()
    else:
        return X_train_full[team_stats_cols].median().tolist()

# Construct test set efficiently
X_test_rows = [
    get_team_stats(season, lower) + get_team_stats(season, higher)
    for season, lower, higher in submission_df[['Season', 'LowerID', 'HigherID']].itertuples(index=False)
]

X_test = pd.DataFrame(X_test_rows, columns=[f'{col}_1' for col in team_stats_cols] + [f'{col}_2' for col in team_stats_cols])

# Check the feature alignment clearly
print("✅ Features prepared for testing.")
print("Features used by model:", rf_model.feature_names_in_)
print("Test DataFrame columns:", X_test.columns.tolist())

missing_in_test = set(rf_model.feature_names_in_) - set(X_test.columns)
extra_in_test = set(X_test.columns) - set(rf_model.feature_names_in_)

print("Missing in test (should be empty):", missing_in_test)
print("Extra in test (should be empty):", extra_in_test)

# Confirm dimensions
print(X_test.shape) # Should clearly show (131407, 20)


In [ ]:
# Rename test dataset columns explicitly to match model
rename_dict = {
    'GamesPlayed_1': 'GamesPlayed',
    'Wins_1': 'Wins',
    'AvgScore_1': 'AvgScore',
    'AvgFGP_1': 'AvgFGP',
    'Avg3PP_1': 'Avg3PP',
    'AvgFT_1': 'AvgFT',
    'AvgORP_1': 'AvgORP',
    'AvgDRP_1': 'AvgDRP',
    'AvgASTTO_1': 'AvgASTTO',
    'AvgNetRtg_1': 'AvgNetRtg',
    'GamesPlayed_2': 'GamesPlayed_L',
    'Wins_2': 'Wins_L',
    'AvgScore_2': 'AvgScore_L',
    'AvgFGP_2': 'AvgFGP_L',
    'Avg3PP_2': 'Avg3PP_L',
    'AvgFT_2': 'AvgFT_L',
    'AvgORP_2': 'AvgORP_L',
    'AvgDRP_2': 'AvgDRP_L',
    'AvgASTTO_2': 'AvgASTTO_L',
    'AvgNetRtg_2': 'AvgNetRtg_L'
}

X_test.rename(columns=rename_dict, inplace=True)

# Final Check for alignment explicitly:
print("✅ Final Feature Alignment Check")
print("Trained model features:", rf_model.feature_names_in_)
print("Test data columns:", X_test.columns.tolist())
print("Mismatches:", set(rf_model.feature_names_in_) - set(X_test.columns))

# Confirm no mismatch
assert set(X_test.columns) == set(rf_model.feature_names_in_), "Mismatch still exists!"

# If no errors raised, proceed to predict
submission_df['Pred'] = rf_model.predict_proba(X_test)[:, 1]
submission_df[['ID', 'Pred']].to_csv('final_submission.csv', index=False)

print("✅ Final submission generated: final_submission.csv")


In [ ]:
# Check for infinite or very large values
print("Infinite values present before replacement:", np.isinf(X_test.values).sum())
print("NaNs before imputation:", X_test.isna().sum().sum())

# Replace infinite values with NaN, then impute
X_test.replace([np.inf, -np.inf], np.nan, inplace=True)
X_test.fillna(X_train_full.median(), inplace=True)

print("Infinite values after replacement:", np.isinf(X_test.values).sum())
print("NaNs after imputation:", X_test.isna().sum().sum())

# Confirm type explicitly to match trained model
X_test = X_test.astype(np.float64)

# Recheck before prediction:
assert np.isinf(X_test.values).sum() == 0, "Still contains infinities!"
assert X_test.isna().sum().sum() == 0, "Still contains NaNs!"

# Now safely predict:
submission_df['Pred'] = rf_model.predict_proba(X_test)[:, 1]

# Save to CSV:
submission_df[['ID', 'Pred']].to_csv('final_submission.csv', index=False)
print("✅ Final submission generated successfully: final_submission.csv")


In [ ]:
print(submission_df[['ID', 'Pred']].head(10))
print(submission_df['Pred'].describe())
print(submission_df['Pred'].isna().sum())



In [ ]:
import pandas as pd
import numpy as np
from sklearn.ensemble import RandomForestClassifier
from sklearn.impute import SimpleImputer
from sklearn.metrics import accuracy_score, brier_score_loss
from sklearn.model_selection import train_test_split

# Load merged dataset
final_merged_path = 'final_merged.csv'
df = pd.read_csv(final_merged_path)

# Define metadata columns to exclude
drop_cols = ['Season', 'DayNum', 'WTeamID', 'LTeamID', 'WScore', 'LScore', 'WLoc', 'NumOT']

# Define features explicitly
features = ['GamesPlayed', 'Wins', 'AvgScore', 'AvgFGP', 'Avg3PP', 'AvgFT',
            'AvgORP', 'AvgDRP', 'AvgASTTO', 'AvgNetRtg',
            'GamesPlayed_L', 'Wins_L', 'AvgScore_L', 'AvgFGP_L', 'Avg3PP_L', 
            'AvgFT_L', 'AvgORP_L', 'AvgDRP_L', 'AvgASTTO_L', 'AvgNetRtg_L']

# Original feature set
X = df[features]
y = np.ones(len(df), dtype=int)

# Create reversed matchups to balance the dataset
df_inv = df.copy()
for col in features:
    if col.endswith('_L'):
        base_col = col[:-2]
        df_inv[col], df_inv[base_col] = df[base_col], df[col]

X_inv = df_inv[features]
y_inv = np.zeros(len(df_inv), dtype=int)

# Combine original and inverted datasets
X_train_full = pd.concat([X, X_inv], ignore_index=True)
y_train_full = np.concatenate([y, y_inv])

# Handle missing or infinite values
X_train_full.replace([np.inf, -np.inf], np.nan, inplace=True)
imputer = SimpleImputer(strategy='median')
X_train_full = pd.DataFrame(imputer.fit_transform(X_train_full), columns=features)

# Train-test split for evaluation
X_train, X_val, y_train_split, y_val = train_test_split(
    X_train_full, y_train_full, test_size=0.2, random_state=42, stratify=y_train_full
)

# Model training
rf_model = RandomForestClassifier(n_estimators=200, max_depth=7,
                                  min_samples_leaf=3, random_state=42, n_jobs=-1)
rf_model.fit(X_train, y_train_split)

# Evaluate on validation set
y_val_pred = rf_model.predict_proba(X_val)[:, 1]
print("Validation Set Performance:")
print(" - Accuracy:", accuracy_score(y_val, (y_val_pred >= 0.5).astype(int)))
print(" - Brier Score:", brier_score_loss(y_val, y_val_pred))

# ------------------- Submission Generation ---------------------
submission_df = pd.read_csv('data/SampleSubmissionStage2.csv')
submission_df[['Season', 'LowerID', 'HigherID']] = submission_df['ID'].str.split('_', expand=True).astype(int)

# Prepare team statistics lookup dictionary
team_stats_cols = ['GamesPlayed', 'Wins', 'AvgScore', 'AvgFGP', 'Avg3PP', 'AvgFT',
                   'AvgORP', 'AvgDRP', 'AvgASTTO', 'AvgNetRtg']

team_stats = {}
for _, row in df.iterrows():
    season = row['Season']
    team_stats[(season, row['WTeamID'])] = row[team_stats_cols].tolist()
    team_stats[(season, row['LTeamID'])] = [row[col + '_L'] for col in team_stats_cols]

# Generate predictions efficiently
predictions = []
for _, row in submission_df.iterrows():
    season, lower, higher = row['Season'], row['LowerID'], row['HigherID']

    # Retrieve stats, fallback to median if team missing
    team1_stats = team_stats.get((season, lower), X_train_full[team_stats_cols].median().tolist())
    team2_stats = team_stats.get((season, higher), X_train_full[team_stats_cols].median().tolist())
    
    match_features = team1_stats + team2_stats

    pred = rf_model.predict_proba([match_features])[0][1]
    predictions.append(pred)

submission_df['Pred'] = predictions
submission_df[['ID', 'Pred']].to_csv('final_submission.csv', index=False)

print("✅ Final submission generated: final_submission.csv")
print(submission_df[['ID', 'Pred']].head())
